# 01 · Reader baselines (untuned Qwen2.5-VL-3B)
Team Dropouts · Correction Inception

## Setup
Kaggle settings: **Accelerator = GPU T4 x2**, **Internet = On**. Add-ons → Secrets → `HF_TOKEN` (you must have accepted FERMAT's terms on Hugging Face).

Add Inputs: datasets **lma-src**, **edu-text**, **edu-circuit-hw-images**, and (to reuse cached grades) the **output of notebook 03**.

In [ ]:
import os, glob, shutil, zipfile, sys
IN = os.environ.get('KAGGLE_INPUT', '/kaggle/input')
WORK = os.environ.get('KAGGLE_WORK', '/kaggle/working')
def find(pattern):
    hits = sorted(glob.glob(f'{IN}/**/{pattern}', recursive=True) + glob.glob(f'{WORK}/_in/**/{pattern}', recursive=True))
    return hits[0] if hits else None
# Kaggle usually unzips uploads; if it kept a .zip, extract it ourselves
if not (find('src/eval/llm.py') and find('edu_data') and find('Screenshot_output_anon/Homework*')):
    for z in glob.glob(f'{IN}/**/*.zip', recursive=True):
        zipfile.ZipFile(z).extractall(f'{WORK}/_in/' + os.path.basename(z)[:-4])
llm, EDU_DATA, EDU_REPO = find('src/eval/llm.py'), find('edu_data'), find('edu_repo')
hw = find('Screenshot_output_anon/Homework*')
IMAGES = os.path.dirname(os.path.dirname(hw)) if hw else None
if not (llm and EDU_DATA and EDU_REPO and IMAGES):
    for root, dirs, files in os.walk(IN):
        if root.count(os.sep) - IN.count(os.sep) < 4: print(root, dirs[:5], files[:5])
    raise SystemExit('Missing inputs: attach lma-src, edu-text AND the images dataset via Add Input. Found: '
                     + str(dict(src=llm, edu_data=EDU_DATA, edu_repo=EDU_REPO, images=IMAGES)))
shutil.copytree(os.path.dirname(os.path.dirname(llm)), f'{WORK}/src', dirs_exist_ok=True)
os.environ.update(EDU_DATA=EDU_DATA, EDU_REPO=EDU_REPO)
os.chdir(WORK)
for rel in ['results/llm_cache.sqlite']:  # reuse grades cached by notebook 03 (attach its output as an input)
    old = find(rel)
    if old and not os.path.exists(f'{WORK}/{rel}'):
        os.makedirs(os.path.dirname(f'{WORK}/{rel}'), exist_ok=True); shutil.copy(old, f'{WORK}/{rel}'); print('reusing cache', old)
DRY = bool(os.environ.get('DRY_JUDGE'))  # local dry run: mock models, no GPU
if not DRY:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    os.system(f'{sys.executable} -m pip install -q -U transformers accelerate peft bitsandbytes qwen-vl-utils datasets python-dotenv')
    os.system('nvidia-smi --query-gpu=name,memory.total --format=csv')
READER = 'mock' if DRY else 'hf:Qwen/Qwen2.5-VL-3B-Instruct'
GRADER = 'mock' if DRY else 'hf-text:Qwen/Qwen2.5-7B-Instruct'
os.environ.update(HF_BATCH='8', HF_TOKEN_BUDGET='16000')
LIMIT = os.environ.get('DRY_LIMIT', '')
ITEMS = 'src/eval/items_with_expert_transcript.csv'  # the 293 exams that have an expert transcript
print('code:', llm, '| data:', EDU_DATA, '| images:', IMAGES)

## 1. Untuned Qwen2.5-VL-3B on FERMAT test: how often does it silently fix student mistakes?

In [ ]:
if not DRY:
    !python src/finetune/eval_fermat.py --model hf:Qwen/Qwen2.5-VL-3B-Instruct --split test --name qwen3b_base --out {WORK}/results/fermat {LIMIT}

## 2. Untuned Qwen2.5-VL-3B transcribes the 293 EDU-CIRCUIT-HW exams that have an expert transcript
Resumable: finished pages are skipped if you re-run.

In [ ]:
import subprocess, time
cmd = (f'{sys.executable} src/eval/run_ocr.py --split observation --items {ITEMS} --task qwen3b_base --model {READER} '
       f'--model-dir qwen3b_base --data-root {IMAGES} --outputs-root {WORK}/Outputs {LIMIT} '
       '--prompt-file src/eval/prompts/Initial_prompt_wo_problem_statement_v6.txt')
procs = []
for g in (0, 1):
    log = open(f'{WORK}/ocr_qwen3b_base_{g}.log', 'w')
    env = {**os.environ, 'CUDA_VISIBLE_DEVICES': str(g)}
    procs.append(subprocess.Popen(cmd + f' --shard {g}/2', shell=True, env=env, stdout=log, stderr=log))
done = lambda: len(glob.glob(f'{WORK}/Outputs/qwen3b_base/**/*_markdown.md', recursive=True))
t0 = time.time()
while any(p.poll() is None for p in procs):
    time.sleep(60 if not DRY else 1)
    m = (time.time() - t0) / 60
    print(f'{m:.0f} min: {done()} pages done', flush=True)
print('exit codes:', [p.returncode for p in procs], '| pages:', done())
for g in (0, 1): print(open(f'{WORK}/ocr_qwen3b_base_{g}.log').read()[-600:])
if any(p.returncode for p in procs): raise SystemExit('a reader process failed: see the log tails above')

### Score `qwen3b_base` transcripts (no LLM judge: Qwen-7B failed as a judge, kappa 0.03)
1) exact maths comparison with the expert transcript; 2) grade impact with the Qwen-7B grader (agrees with TAs 71%; expert-transcript grades are reused from notebook 03's cache).

In [ ]:
!python src/eval/transcript_diff.py --target-root {WORK}/Outputs/qwen3b_base --target-model-dir qwen3b_base --name qwen3b_base
!python src/eval/grade_impact.py --target-root {WORK}/Outputs/qwen3b_base --target-model-dir qwen3b_base --grader {GRADER} --name B1_qwen3b_base_qwen7b --workers 16 {LIMIT}

## 3. Download `reader_baselines.zip` from the Output tab

In [ ]:
# results + transcripts (markdown only, not the copied page images)
with zipfile.ZipFile(f'{WORK}/reader_baselines.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f'{WORK}/results/**/*', recursive=True) + glob.glob(f'{WORK}/Outputs/**/*.md', recursive=True):
        if os.path.isfile(f): z.write(f, os.path.relpath(f, WORK))
print(os.path.getsize(f'{WORK}/reader_baselines.zip') // 1024, 'KB')